# Task 2: Customer Segmentation Using Unsupervised Learning
**DevelopersHub Corporation – Data Science Internship**

## Problem Statement
Cluster mall customers based on their spending habits and demographic data, then propose targeted marketing strategies for each segment.

## Objective
- Perform EDA on Mall Customers dataset
- Apply K-Means Clustering
- Visualize clusters using PCA and t-SNE
- Suggest marketing strategies per cluster

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

print('All libraries imported successfully!')

## 2. Load Dataset
> Download from: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python  
> File: `Mall_Customers.csv`

In [ ]:
# Load dataset
df = pd.read_csv('Mall_Customers.csv')
print('Shape:', df.shape)
df.head(10)

## 3. Dataset Description

In [ ]:
# Dataset info
print('=== Dataset Info ===')
df.info()

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
# Rename columns for convenience
df.columns = ['CustomerID', 'Gender', 'Age', 'AnnualIncome', 'SpendingScore']
print('Column names:', df.columns.tolist())
print('\nGender distribution:')
print(df['Gender'].value_counts())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes,
                           ['Age', 'AnnualIncome', 'SpendingScore'],
                           ['steelblue', 'coral', 'green']):
    ax.hist(df[col], bins=20, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('distributions.png', dpi=150)
plt.show()

In [ ]:
# Gender distribution pie chart
plt.figure(figsize=(5, 5))
df['Gender'].value_counts().plot(
    kind='pie', autopct='%1.1f%%',
    colors=['lightcoral', 'steelblue'],
    startangle=90, explode=[0.05, 0.05]
)
plt.title('Gender Distribution')
plt.ylabel('')
plt.tight_layout()
plt.savefig('gender_distribution.png', dpi=150)
plt.show()

In [ ]:
# Boxplots by Gender
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['Age', 'AnnualIncome', 'SpendingScore']):
    df.boxplot(column=col, by='Gender', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col} by Gender')
    ax.set_xlabel('Gender')

plt.suptitle('Feature Distributions by Gender', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_by_gender.png', dpi=150)
plt.show()

In [ ]:
# Scatter: Income vs Spending Score
plt.figure(figsize=(8, 6))
colors = df['Gender'].map({'Male': 'steelblue', 'Female': 'coral'})
plt.scatter(df['AnnualIncome'], df['SpendingScore'],
            c=colors, alpha=0.7, edgecolors='black', s=80)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Annual Income vs Spending Score')
# Manual legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Male'),
                   Patch(facecolor='coral', label='Female')]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('income_vs_spending.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
le = LabelEncoder()
df_corr = df.copy()
df_corr['Gender'] = le.fit_transform(df_corr['Gender'])
sns.heatmap(df_corr.drop('CustomerID', axis=1).corr(),
            annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('corr_heatmap.png', dpi=150)
plt.show()

## 5. Preprocessing for Clustering

In [ ]:
# Select features for clustering
features = ['Age', 'AnnualIncome', 'SpendingScore']
X = df[features].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature matrix shape:', X_scaled.shape)

## 6. Finding Optimal K – Elbow Method & Silhouette Score

In [ ]:
# Elbow method
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=5, color='red', linestyle='--', label='K=5 (optimal)')
axes[0].legend()

# Silhouette scores
axes[1].plot(K_range, silhouette_scores, 'gs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')
axes[1].axvline(x=5, color='red', linestyle='--', label='K=5 (optimal)')
axes[1].legend()

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150)
plt.show()

print(f'Best Silhouette Score at K=5: {silhouette_scores[3]:.4f}')

## 7. K-Means Clustering with K=5

In [ ]:
# Apply K-Means with K=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Cluster summary statistics
cluster_summary = df.groupby('Cluster')[features].mean().round(2)
cluster_summary['Size'] = df['Cluster'].value_counts().sort_index()
print('=== Cluster Summary ===')
print(cluster_summary)

In [ ]:
# 2D Scatter: Income vs Spending Score colored by cluster
colors_map = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']
segment_names = ['Careful Spenders', 'Standard Customers',
                 'Target Customers', 'Careless Spenders', 'Sensible Customers']

plt.figure(figsize=(10, 7))
for cluster in range(5):
    mask = df['Cluster'] == cluster
    plt.scatter(
        df[mask]['AnnualIncome'],
        df[mask]['SpendingScore'],
        c=colors_map[cluster],
        s=100, alpha=0.8, edgecolors='black',
        label=f'Cluster {cluster}: {segment_names[cluster]}'
    )

# Plot centroids
centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 1], centers[:, 2],
            marker='*', s=300, c='black', zorder=5, label='Centroids')

plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.title('Customer Segments – K-Means (K=5)', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150)
plt.show()

## 8. PCA Visualization

In [ ]:
# PCA: Reduce to 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'PCA Explained Variance: {pca.explained_variance_ratio_}')
print(f'Total Variance Explained: {sum(pca.explained_variance_ratio_):.4f}')

plt.figure(figsize=(9, 7))
for cluster in range(5):
    mask = df['Cluster'] == cluster
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors_map[cluster], s=80,
        alpha=0.8, edgecolors='black',
        label=f'Cluster {cluster}: {segment_names[cluster]}'
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.title('PCA Cluster Visualization', fontsize=13, fontweight='bold')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150)
plt.show()

## 9. t-SNE Visualization

In [ ]:
# t-SNE: Reduce to 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(9, 7))
for cluster in range(5):
    mask = df['Cluster'] == cluster
    plt.scatter(
        X_tsne[mask, 0], X_tsne[mask, 1],
        c=colors_map[cluster], s=80,
        alpha=0.8, edgecolors='black',
        label=f'Cluster {cluster}: {segment_names[cluster]}'
    )

plt.xlabel('t-SNE Dimension 1', fontsize=11)
plt.ylabel('t-SNE Dimension 2', fontsize=11)
plt.title('t-SNE Cluster Visualization', fontsize=13, fontweight='bold')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig('tsne_clusters.png', dpi=150)
plt.show()

## 10. Marketing Strategy for Each Segment

In [ ]:
# Radar / bar chart showing cluster profiles
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(3)
width = 0.15
bar_features = ['Age', 'AnnualIncome', 'SpendingScore']

for i, cluster in enumerate(range(5)):
    vals = cluster_summary.loc[cluster, bar_features].values
    ax.bar(x + i*width, vals, width, label=f'C{cluster}: {segment_names[cluster]}',
           color=colors_map[cluster], alpha=0.85, edgecolor='black')

ax.set_xticks(x + 2*width)
ax.set_xticklabels(['Age', 'Annual Income (k$)', 'Spending Score'], fontsize=11)
ax.set_ylabel('Mean Value')
ax.set_title('Cluster Profiles – Mean Features per Segment', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=150)
plt.show()

## 11. Final Conclusion & Marketing Strategies

### Cluster Profiles & Marketing Strategies:

| Cluster | Segment Name | Profile | Marketing Strategy |
|---------|-------------|---------|--------------------|
| **0** | Careful Spenders | Low Income, Low Spending Score | Offer discounts, budget-friendly products, loyalty reward programs to increase engagement |
| **1** | Standard Customers | Medium Income, Medium Spending | Send personalized newsletters, moderate promotions, cross-sell complementary items |
| **2** | Target Customers ⭐ | High Income, High Spending Score | Premium membership perks, exclusive launches, VIP events – highest ROI segment |
| **3** | Careless Spenders | Low Income, High Spending Score | Offer installment plans, credit-based purchases, budget management tools |
| **4** | Sensible Customers | High Income, Low Spending Score | Targeted luxury campaigns, highlight value & quality, build trust through premium positioning |

### Key Insights:
- **Cluster 2 (Target Customers)** is the most profitable — high income AND high spending. Marketing budget should prioritize this segment.
- **Cluster 4 (Sensible Customers)** have money but spend less — opportunity to convert them with premium quality messaging.
- **PCA** explained ~80%+ of variance in 2 components, validating the feature selection.
- **t-SNE** shows clear, well-separated clusters — confirming good cluster quality.
- **Silhouette Score** at K=5 confirms optimal segmentation.